# 23. Ders: Seslerin Matematiği (Konuşmayı Metne Çevirme / ASR)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

"Hey Siri, yarın saat 7'ye alarm kur" dediğinizde, cebinizdeki telefon söylediğiniz şeyin 'alarm' kelimesi olduğunu nereden anlıyor? Ses dediğimiz şey aslında havada yayılan titreşimlerdir (dalgalardır). Telefonunuzun mikrofonu bu dalgaları yakalar ve onları (tıpkı resimlerdeki pikseller gibi) devasa bir sayı tablosuna dönüştürür.

Yapay zeka modelleri (özellikle OpenAI'ın Whisper'ı gibi modern sistemler), bu ses dalgası rakamlarına bakar ve daha önceden dinlediği milyonlarca saatlik ses kayıtlarıyla kıyaslayarak hangi saniyenin hangi harfe (veya kelimeye) denk geldiğini tahmin eder. Bu teknolojiye Otomatik Konuşma Tanıma (Automatic Speech Recognition - ASR) diyoruz. Bugün kendi toplantı deşifre veya asistan botumuzu yazacağız.

## Adım 2: Teknik Terimler ve Sözlük

- ASR (Automatic Speech Recognition): İnsan konuşmasını dinleyip otomatik olarak metne (text) döken yapay zeka sistemleri.
- Audio Waveform (Ses Dalgası): Sesin bilgisayardaki görsel ve matematiksel (sayısal) frekans karşılığı.
- Transcript (Deşifre / Döküm): Bir videodaki veya ses kaydındaki konuşmaların eksiksiz yazılı metin hali (Örn: YouTube Altyazıları).
- Whisper: ChatGPT'nin üreticisi OpenAI tarafından geliştirilen ve dünyanın hemen her dilindeki sesleri mükemmel bir şekilde metne çevirebilen ünlü açık kaynaklı model.

## Adım 3: Ses İşleme İçin Hugging Face ASR Pipeline

Önceki derslerde Metin (`text-generation`), Resim (`image-classification`) işledik. Şimdi `automatic-speech-recognition` pipeline'ını kullanacağız.

In [ ]:
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

# OpenAI'ın hafif ve çok hızlı çalışan 'whisper-tiny' modelini indiriyoruz.
# Tiny modeli küçük olduğu için Türkçe başarısı bazen düşük olabilir, ama İngilizcede kusursuzdur.
kulak_modeli = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")

print("Yapay Zeka kulaklarını açtı, sesleri dinlemeye hazır.")

## Adım 4: İnternetten Örnek Bir Ses Dosyası (WAV/MP3) Yükleme

Test etmek için kısa bir ses dosyasına ihtiyacımız var. İnternetten ücretsiz, İngilizce bir 'Hello world' (Merhaba dünya) ses kaydı indirelim.

In [ ]:
import urllib.request

# İnternetteki açık kaynaklı bir ses test dosyası
ses_url = "https://www.signalogic.com/melp/EngSamples/Orig/eng_m1.wav"
ses_dosyasi = "test_sesi.wav"

# Dosyayı indirip Colab içine kaydediyoruz
urllib.request.urlretrieve(ses_url, ses_dosyasi)

print("Ses dosyası (test_sesi.wav) bilgisayara/sunucuya indirildi.")

## Adım 5: Gelen Sesi IPython İle Colab'de Dinleme

İndirdiğimiz dosyanın içinde ne söylendiğini kodlarımızda bir medya oynatıcı oluşturarak bizzat dinleyelim.

In [ ]:
from IPython.display import Audio

# Jupyter/Colab üzerinde ses dosyalarını oynatan sihirli bir widget (araç)
display(Audio(ses_dosyasi))

# Oynat (Play) tuşuna basarak adamın ne dediğini dinleyin. (Yaklaşık: "A pot of tea helps to calm the nerves...")

## Adım 6: Sesi Yapay Zekaya (Whisper) Dinletme ve Transcript Alma

Şimdi aynı dosyayı kulak_modelimize (Whisper) vereceğiz. Bakalım bizim duyduğumuzu doğru bir şekilde yazıya (text) dökebilecek mi?

In [ ]:
print("Model sesi analiz ediyor, lütfen bekleyin...")

# pipeline, ses dosyasının yolunu (ismini) doğrudan kabul eder.
ses_ceviri_sonucu = kulak_modeli(ses_dosyasi)

print("\n--- YAPAY ZEKA DEŞİFRE METNİ (TRANSCRIPT) ---")
# Çıktı yine bir sözlük (dictionary) şeklindedir, 'text' anahtarını çekeriz.
print(ses_ceviri_sonucu['text'])

## Adım 7: Gürültülü Seslerde Yapay Zekanın Hata Yapması

Eğer ortamda rüzgar sesi, müzik veya iki kişi aynı anda konuşuyorsa model kelimeleri yanlış anlayabilir (Tıpkı Siri'nin bazen bizi yanlış anlaması gibi). Bu durumlarda arka plan gürültüsünü temizleyen ayrı sinyal işleme algoritmaları (Örn: librosa kütüphanesi) kullanılarak ses önce 'yıkanır/temizlenir', sonra Whisper'a verilir.

## Adım 8: Dene ve Keşfet - Kendi Ses Kaydını Test Et

Telefonunuzun ses kaydedicisiyle 5-10 saniyelik İngilizce (veya Türkçe, ancak tiny modelde Türkçe hata yapabilir) bir ses kaydedin. Dosyayı `.wav` veya `.mp3` olarak Colab'in yan tarafındaki klasör (Dosyalar) ikonuna tıklayarak sürükleyip bırakın. 

Daha sonra Adım 6'daki `kulak_modeli("test_sesi.wav")` kısmını `kulak_modeli("sizin_dosyaniz.mp3")` olarak değiştirip, yapay zekanın sizi ne kadar iyi anladığını test edin.

## Adım 9: Sıra Sende Görevi - Toplantı Özetleyici Asistan (Birleştirme)

Problem: Yönetim kurulu toplantınızın 2 saatlik bir ses kaydı var. Bu sesi önce metne (transcript) çeviren, ardından (14. Derste öğrendiğimiz) Summarization modeli ile bu metni tek paragraflık bir özete çeviren süper bir asistan yazmanız isteniyor.

Gereksinimler:
1. Adım 3'te kurduğumuz `kulak_modeli`'nin yanına, 14. dersteki gibi bir `ozetleyici_model = pipeline("summarization")` tanımlayın.
2. `toplanti_asistani(ses_yolu)` adında bir fonksiyon oluşturun.
3. Fonksiyon içinde önce sesi `kulak_modeli`ne verip uzun metni (transcript) çekin ve `uzun_metin` değişkenine atayın.
4. Daha sonra bu `uzun_metin`i `ozetleyici_model` içine gönderip `max_length=30` parametresiyle kısa özetini (`kisa_ozet`) alın.
5. Ekrana hem sesi yazıya dökülmüş tam metnini hem de YZ Özetini alt alta şık bir biçimde yazdırın.
6. Dışarıda fonksiyonu `toplanti_asistani(ses_dosyasi)` diyerek test_sesi.wav dosyası ile çalıştırın.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 23. Dersi tamamlayarak sesi metne döken algoritmaların (Speech Recognition) çalışma mantığını kavradınız ve farklı AI yeteneklerini (ASR ve Summarization) Lego parçaları gibi birleştirip (Pipeline zincirleme) gerçek dünyada milyonlarca dolara satılan B2B yazılımlarının çekirdeğini yazdınız.

Bir sonraki derste sürecin tam tersini yapacağız: Bilgisayara biz metin vereceğiz ve onun (robot gibi değil, bir insan gibi) konuşmasını (Text-to-Speech) sağlayacağız.